# Exercises XP – Week 6, Day 3: Time Series with LSTM
Dataset: **Household Electric Power Consumption** (one household, one measurement per minute, Dec 2006 – Nov 2010).
⚠️ Run in **Google Colab with a GPU** (Runtime → Change runtime type → GPU). The notebook downloads the dataset itself.

## Part 1: Data Import and Initial Exploration

In [ ]:
import os, glob, zipfile, urllib.request
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

URL = "https://github.com/devtlv/Datasets-GEN-AI-Bootcamp/raw/refs/heads/main/Week%206/W6D4/household_power_consumption.zip"
if not glob.glob("power_data/**/*.txt", recursive=True) and not glob.glob("power_data/**/*.csv", recursive=True):
    urllib.request.urlretrieve(URL, "power.zip")
    with zipfile.ZipFile("power.zip") as z:
        z.extractall("power_data")

data_file = (glob.glob("power_data/**/*.txt", recursive=True) + glob.glob("power_data/**/*.csv", recursive=True))[0]
print("Reading:", data_file)

# The file uses ";" as separator and "?" for missing values
df = pd.read_csv(data_file, sep=";", na_values=["?", "nan"], low_memory=False)
df.head()

In [ ]:
print("Shape:", df.shape)
df.dtypes

In [ ]:
# Combine Date + Time into one datetime index (dates are written day/month/year)
df["datetime"] = pd.to_datetime(df["Date"] + " " + df["Time"], format="%d/%m/%Y %H:%M:%S")
df = df.drop(columns=["Date", "Time"]).set_index("datetime")

numeric_cols = df.columns
df[numeric_cols] = df[numeric_cols].apply(pd.to_numeric, errors="coerce")
print(df.dtypes)
print("From", df.index.min(), "to", df.index.max())

## Part 2: Handling Missing Values

In [ ]:
missing = df.isnull().sum()
print("Missing values per column:")
print(missing)
print(f"\nRows with missing values: {df.isnull().any(axis=1).sum():,} of {len(df):,} ({df.isnull().any(axis=1).mean():.2%})")

In [ ]:
df = df.fillna(df.mean())
print("Missing values after filling with the column means:", df.isnull().sum().sum())

*(Filling with the overall mean is what the exercise asks for. For time series, filling from neighbouring time points, e.g. interpolation, is often more realistic, since the missing values come in blocks of hours or days.)*

## Part 3: Data Visualization

In [ ]:
daily_power = df["Global_active_power"].resample("D").agg(["sum", "mean"])

fig, axes = plt.subplots(2, 1, figsize=(14, 7), sharex=True)
daily_power["sum"].plot(ax=axes[0], color="steelblue")
axes[0].set_title("Global active power – daily SUM (kW-minutes)")
daily_power["mean"].plot(ax=axes[1], color="darkorange")
axes[1].set_title("Global active power – daily MEAN (kW)")
plt.tight_layout()
plt.show()

In [ ]:
daily_intensity = df["Global_intensity"].resample("D").agg(["mean", "std"])

plt.figure(figsize=(14, 4))
plt.plot(daily_intensity.index, daily_intensity["mean"], label="daily mean", color="seagreen")
plt.fill_between(daily_intensity.index,
                 daily_intensity["mean"] - daily_intensity["std"],
                 daily_intensity["mean"] + daily_intensity["std"],
                 alpha=0.25, color="seagreen", label="± 1 standard deviation")
plt.title("Global intensity (amperes) – daily mean and standard deviation")
plt.legend()
plt.show()

**What the charts show:** consumption follows a clear **yearly seasonal pattern**: it's higher in winter (heating, lighting) and lower in summer, with a dip each August (likely holidays). Global intensity moves almost exactly with active power, since current and power are closely linked. The daily standard deviation is large, because usage swings a lot within a day (cooking, appliances).

## Part 4: Data Preprocessing for LSTM
The minute-level data has ~2 million rows, which is too much for a quick LSTM exercise, so we **resample to hourly averages** (~35,000 rows).
**Task:** use the previous **24 hours** of all measurements to predict the **next hour's Global_active_power**.

In [ ]:
from sklearn.preprocessing import MinMaxScaler

hourly = df.resample("h").mean()
print("Hourly data:", hourly.shape)

# Normalize to [0, 1] – the scaler is fitted on the training period only, to avoid leakage
TARGET = "Global_active_power"
n_train = int(len(hourly) * 0.8)          # first 80% of time = training, last 20% = testing (no shuffling!)

scaler = MinMaxScaler()
scaler.fit(hourly.iloc[:n_train])
scaled = scaler.transform(hourly)
target_idx = list(hourly.columns).index(TARGET)
print("Features:", list(hourly.columns))

In [ ]:
def make_sequences(data, window=24):
    """Turn a 2D array (time, features) into LSTM inputs (samples, window, features) and next-step targets."""
    X, y = [], []
    for i in range(window, len(data)):
        X.append(data[i - window:i])
        y.append(data[i, target_idx])
    return np.array(X), np.array(y)

WINDOW = 24
X_all, y_all = make_sequences(scaled, WINDOW)

split = n_train - WINDOW
X_train, y_train = X_all[:split], y_all[:split]
X_test, y_test = X_all[split:], y_all[split:]

print("X_train:", X_train.shape, "→ (samples, time steps, features)")
print("X_test: ", X_test.shape)

## Part 5: Building an LSTM Model

In [ ]:
import tensorflow as tf
from tensorflow.keras import layers, models

tf.random.set_seed(42)

model = models.Sequential([
    layers.Input(shape=(X_train.shape[1], X_train.shape[2])),
    layers.LSTM(64, return_sequences=True),   # first LSTM layer passes its full sequence to the next
    layers.Dropout(0.2),
    layers.LSTM(32),                          # second LSTM layer summarises the sequence
    layers.Dropout(0.2),
    layers.Dense(1),                          # one number: next hour's (scaled) active power
])
model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=0.001), loss="mse", metrics=["mae"])
model.summary()

**Architecture choices:** two stacked LSTM layers (64 and 32 units) can learn both short-term patterns (time of day) and longer dependencies within the 24-hour window. **Dropout** reduces overfitting. A single linear **Dense(1)** output fits a regression task. **MSE loss** with the **Adam** optimizer is the standard choice for predicting continuous values.

## Part 6: Training and Evaluating the LSTM Model

In [ ]:
early_stop = tf.keras.callbacks.EarlyStopping(monitor="val_loss", patience=3, restore_best_weights=True)
history = model.fit(X_train, y_train, epochs=20, batch_size=64,
                    validation_split=0.1, shuffle=False, callbacks=[early_stop])

In [ ]:
test_loss, test_mae = model.evaluate(X_test, y_test, verbose=0)
print(f"Test MSE (scaled): {test_loss:.5f} | Test MAE (scaled): {test_mae:.4f}")

# Convert predictions back to kilowatts
def inverse_target(values):
    dummy = np.zeros((len(values), scaled.shape[1]))
    dummy[:, target_idx] = values.ravel()
    return scaler.inverse_transform(dummy)[:, target_idx]

pred_kw = inverse_target(model.predict(X_test, verbose=0))
true_kw = inverse_target(y_test)

from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
print(f"MAE: {mean_absolute_error(true_kw, pred_kw):.3f} kW | RMSE: {np.sqrt(mean_squared_error(true_kw, pred_kw)):.3f} kW | R²: {r2_score(true_kw, pred_kw):.3f}")

# Naive baseline: "next hour = this hour"
naive = inverse_target(X_test[:, -1, target_idx])
print(f"Naive baseline MAE: {mean_absolute_error(true_kw, naive):.3f} kW")

In [ ]:
plt.figure(figsize=(9, 4))
plt.plot(history.history["loss"], label="training loss")
plt.plot(history.history["val_loss"], label="validation loss")
plt.title("Training and validation loss (MSE)")
plt.xlabel("Epoch")
plt.legend()
plt.show()

plt.figure(figsize=(14, 4))
plt.plot(true_kw[:24 * 7], label="actual")
plt.plot(pred_kw[:24 * 7], label="LSTM prediction")
plt.title("One week of the test period – hourly global active power (kW)")
plt.xlabel("Hour")
plt.legend()
plt.show()

**How to interpret the results:**
- **Loss curves:** both losses should drop quickly in the first epochs and then flatten. If validation loss starts rising while training loss keeps falling, the model is overfitting. Early stopping then restores the best epoch.
- **Prediction plot:** a good model follows the daily rhythm (morning and evening peaks, low usage at night). It usually smooths out sudden spikes, such as an appliance switching on, which are hard to predict from past values alone.
- **Compare with the naive baseline** ("next hour = current hour"). The LSTM is only useful if its MAE is clearly lower than the baseline's. Household electricity is noisy, so even a good model has a noticeable error.